# 03 · Identifiers and delistings: join on CIK, keep the dead companies

Two habits quietly break stock research:

1. **Joining on ticker.** Tickers get renamed, reused and moved between legal entities.
2. **Using today's list of companies.** Companies that went bankrupt or were acquired
   drop out, so the history that's left is the survivors (survivorship bias).

The SEC's CIK (Central Index Key) is permanent for a legal entity, Varve's `varve_id` is
permanent for a company (even when it moves to a new legal entity), and Varve keeps every
company that ever filed, including the dead ones. The free sample includes some
deliberately awkward cases.

In [1]:
import sys
sys.path.append("..")

import pandas as pd
from varve_sample import load

pd.set_option("display.width", 200, "display.max_columns", 20, "display.max_colwidth", 70)
companies = load("fundamentals_companies")
q = load("fundamentals_quarterly")
a = load("fundamentals_annual")

## One ticker, two companies

In [2]:
companies["tickers"] = companies["tickers_all"].str.split("|")
shared = companies.explode("tickers").groupby("tickers").filter(lambda g: g["cik"].nunique() > 1)
shared.sort_values(["tickers", "first_filed"])[
    ["tickers", "cik", "name", "former_names", "first_filed", "last_filed"]]

,tickers,cik,name,former_names,first_filed,last_filed
14,BBBY,886158,BED BATH & BEYOND INC,NaN,2009-10-07,2023-06-14
20,BBBY,1130713,"BED BATH & BEYOND, INC.","BEYOND, INC.|OVERSTOCK COM INC|OVERSTOCK.COM, INC",2011-07-28,2026-08-04
15,DIS,1001039,WALT DISNEY CO/,DC HOLDCO INC,2009-07-30,2019-02-05
26,DIS,1744489,WALT DISNEY CO,TWDC HOLDCO 613 CORP.,2019-05-08,2026-08-05
21,GOOG,1288776,GOOGLE INC.,GOOGLE INC.,2009-08-04,2015-10-29
25,GOOG,1652044,ALPHABET INC.,NaN,2015-10-29,2026-07-23
21,GOOGL,1288776,GOOGLE INC.,GOOGLE INC.,2009-08-04,2015-10-29
25,GOOGL,1652044,ALPHABET INC.,NaN,2015-10-29,2026-07-23


* **BBBY**: Bed Bath & Beyond (CIK 886158) went bankrupt in 2023. Overstock (CIK
  1130713) bought the brand, took the Bed Bath & Beyond name and has used the BBBY
  symbol too: a different company with different financials under the same name and
  ticker.
* **GOOGL**: Google Inc. (1288776) became a subsidiary of the new Alphabet Inc.
  (1652044) in 2015. **DIS**: likewise for Disney in 2019, during the Fox deal. Each
  reorganization started a new CIK, so a long history spans two (next section).

A ticker join attaches one company's numbers to the other. Even taking only the
"current" holder of each ticker loses the old company's history, or mixes the two.

## One company, two CIKs: `varve_id`

When a company reorganizes under a new holding company or reincorporates, the new legal
entity gets a new CIK and the old one stops reporting. `varve_id` follows the company:
it's "VD" plus the CIK of the first entity in the chain, and it's in every table.
`company_lineage` lists each chain, with the dates and the evidence for the link (the
successor-issuer notice the new entity filed, and the ticker moving across).

In [3]:
lineage = load("company_lineage")
lineage[["varve_id", "seq", "cik", "name", "valid_from", "valid_to", "link_evidence", "in_dataset"]]

,varve_id,seq,cik,name,valid_from,valid_to,link_evidence,in_dataset
0,VD1001039,1,1001039,TWDC Enterprises 18 Corp.,1996-02-14,2019-03-19,NaN,True
1,VD1001039,2,1744489,Walt Disney Co,2019-03-20,NaT,ticker handoff DIS,True
2,VD1288776,1,1288776,GOOGLE INC.,2004-08-16,2015-10-01,NaN,True
3,VD1288776,2,1652044,Alphabet Inc.,2015-10-02,NaT,ticker handoff GOOG + named in notice,True
4,VD34088,1,34088,EXXON MOBIL CORP,1994-03-11,2026-06-30,NaN,True
5,VD34088,2,2115436,ExxonMobil Holdings Corp,2026-07-01,NaT,ticker handoff XOM,False


Exxon's move to a holding company in July 2026 is already linked, before the new entity
has filed any financials of its own (`in_dataset` is False for it).

To get one continuous series per company, key on `varve_id` and keep the
`lineage_primary` rows. Around a reorganization both entities can report the same
period: the new entity's first annual report repeats the old one's figures as
comparatives, filed later. `lineage_primary` keeps the row filed first, so the series
stays point-in-time and has exactly one row per period.

In [4]:
googl = a[(a["varve_id"] == "VD1288776") & a["lineage_primary"]].sort_values("period_end")
pd.DataFrame({
    "period_end": googl["period_end"].dt.date,
    "cik": googl["cik"],
    "filed": googl["filed"].dt.date,
    "revenue_B": googl["revenue"] / 1e9,
}).query("period_end.astype('str').between('2012', '2018')", engine="python").round(1)

,period_end,cik,filed,revenue_B
455,2012-12-31,1288776,2015-02-09,50.2
456,2013-12-31,1288776,2015-02-09,59.8
457,2014-12-31,1288776,2015-02-09,66.0
503,2015-12-31,1652044,2016-05-03,75.0
504,2016-12-31,1652044,2019-02-05,90.3
505,2017-12-31,1652044,2019-02-05,110.9


Google Inc. (CIK 1288776) reports through fiscal 2014, Alphabet (CIK 1652044) from 2015,
under one id.

Not every new CIK is a continuation, so these are deliberately **not** linked: a new
company formed by merging two, acquisitions, spin-offs, blank-check (SPAC) mergers, and a
name or ticker passing to a different company. Overstock and the bankrupt Bed Bath &
Beyond keep different ids:

In [5]:
companies[companies["cik"].isin([886158, 1130713])][["cik", "varve_id", "name", "tickers_all"]]

,cik,varve_id,name,tickers_all
14,886158,VD886158,BED BATH & BEYOND INC,BBBY|BBBYQ
20,1130713,VD1130713,"BED BATH & BEYOND, INC.",BBBY|BBBY-WT|BBBYW|BYON|NXH|OSTBP|OSTK|OSTKO


## Names and tickers over time

`former_names` and `tickers_all` are summaries. The dated histories behind them are in
`company_names` and `ticker_history`: here's CIK 1130713 going from Overstock to Beyond to
Bed Bath & Beyond, and its symbols with the dates each was in use.

In [6]:
names = load("company_names")
names[names["cik"] == 1130713][["cik", "name", "valid_from", "valid_to"]]

,cik,name,valid_from,valid_to
25,1130713,"OVERSTOCK.COM, INC",2011-07-28,2024-02-23
26,1130713,"BEYOND, INC.",2024-02-23,2025-10-27
27,1130713,"BED BATH & BEYOND, INC.",2025-10-27,NaT


In [7]:
tickers = load("ticker_history")
tickers[tickers["cik"] == 1130713][["cik", "ticker", "valid_from", "valid_to", "current"]]

,cik,ticker,valid_from,valid_to,current
46,1130713,OSTK,2006-02-17,2023-11-02,False
47,1130713,OSTBP,2021-06-03,2022-03-01,False
48,1130713,OSTKO,2021-06-03,2022-07-03,False
49,1130713,BYON,2023-11-13,2025-08-19,False
50,1130713,BBBY,2025-11-18,2026-09-08,False
51,1130713,BBBY-WT,2025-12-06,2026-08-01,False
52,1130713,BBBYW,2026-09-08,NaT,True
53,1130713,NXH,2026-09-08,NaT,True


## Renamed tickers

`tickers_all` lists every symbol recovered for a CIK. Joining on CIK is unaffected when
the symbol changes.

In [8]:
companies[companies["ticker"].isin(["META", "BKNG", "NFLX"])][
    ["cik", "ticker", "tickers_all", "name", "former_names"]]

,cik,ticker,tickers_all,name,former_names
18,1065280,NFLX,NFLX|NLFX,NETFLIX INC,NETFLIX COM INC
19,1075531,BKNG,BKNG|PCLN,BOOKING HOLDINGS INC.,PRICELINE COM INC|PRICELINE GROUP INC.
23,1326801,META,FB|META,"META PLATFORMS, INC.",FACEBOOK INC


Price data you buy elsewhere is usually keyed by ticker and date. Map it to CIK (and from
there to `varve_id`) with the ticker that was in use **on that date**, using
`ticker_history` (above).

## Companies that no longer exist

`ticker_current` is empty when the SEC's current ticker list has no symbol for the
company. Their data stays in the dataset.

In [9]:
gone = companies[companies["ticker_current"].isna()].sort_values("last_filed")
gone[["cik", "ticker", "name", "last_filed"]]

,cik,ticker,name,last_filed
21,1288776,GOOGL,GOOGLE INC.,2015-10-29
15,1001039,DIS,WALT DISNEY CO/,2019-02-05
24,1418091,TWTR,"TWITTER, INC.",2022-07-26
12,719739,SIVB,SVB FINANCIAL GROUP,2023-02-24
14,886158,BBBY,BED BATH & BEYOND INC,2023-06-14
3,34088,XOM,EXXON MOBIL CORP,2026-08-03


Most of these are the familiar endings: SVB Financial collapsed in March 2023, Bed
Bath & Beyond filed for bankruptcy a month later, Twitter was taken private in 2022, and
the old Google and Disney entities were replaced by new parents.

Exxon Mobil is the odd one out: it still files, but XOM moved to its new holding
company, ExxonMobil Holdings Corp (CIK 2115436), so the old entity has no current ticker.
It isn't dead: the lineage above links the two under `VD34088`.

## What the last filings looked like

Survivors-only data never shows a company on its way down. Here are SVB's last
quarters: total assets peaked in early 2022 and shrank through the year while quarterly
profit fell. Its last 10-K was filed on 24 February 2023, two weeks before the bank
failed.

In [10]:
svb = q[q["cik"] == 719739].sort_values("period_end")
pd.DataFrame({
    "period_end": svb["period_end"].dt.date,
    "filed": svb["filed"].dt.date,
    "total_assets_B": svb["total_assets"] / 1e9,
    "equity_B": svb["equity"] / 1e9,
    "net_income_M": svb["net_income"] / 1e6,
}).tail(6).round(1)

,period_end,filed,total_assets_B,equity_B,net_income_M
929,2021-09-30,2021-11-08,191.0,14.3,475.0
930,2021-12-31,2022-03-01,211.5,16.2,408.2
931,2022-03-31,2022-05-06,220.4,16.0,533.0
932,2022-06-30,2022-08-08,214.4,15.9,353.0
933,2022-09-30,2022-11-07,212.9,15.5,418.0
934,2022-12-31,2023-02-24,211.8,16.0,305.0


A screen run on today's companies can never select SVB, Bed Bath & Beyond or Twitter,
so a backtest built on it never takes their losses. With point-in-time data, a 2022
screen sees them exactly as investors did.

## Next

**04 · Events**: earnings release times, insider buying and 13D activists, from the
EDGAR daily feed.